# Supply Chain Cost Intelligence — Notebook 01: EDA

**Dataset:** USAspending.gov federal procurement (FY2023, contracts, chilled to a manageable NAICS scope)

**Goals:**
- Understand spend distribution, vendor concentration, category structure
- Pareto analysis: which 20% of vendors account for 80% of spend?
- Lead time distribution: what does normal look like?
- Identify the NAICS categories with the most analytical leverage
- Lay the groundwork for SQL analysis and clustering

**Run first:** `conda activate supply-chain` then `python -m src.data_loader`

In [ ]:
import sys; sys.path.insert(0, '..')
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path
from src.data_loader import DB_PATH, get_awards_summary, get_top_naics

sns.set_theme(style='darkgrid')
figures = Path('../figures')
figures.mkdir(exist_ok=True)

conn = duckdb.connect(str(DB_PATH), read_only=True)
print('Connected to DuckDB')

## 1. Dataset Overview

In [ ]:
summary = get_awards_summary(conn)
print(summary.to_string())

## 2. Spend Distribution

Expect heavily right-skewed: most awards small, a few extremely large contracts.
Log-transform will reveal the full distribution.

In [ ]:
amounts = conn.execute('SELECT action_amount FROM federal_awards WHERE action_amount > 0').df()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
amounts['action_amount'].clip(upper=amounts['action_amount'].quantile(0.99)).hist(bins=60, ax=axes[0])
axes[0].set_title('Award Amounts (clip p99)')
axes[0].set_xlabel('USD')
np.log10(amounts['action_amount'] + 1).hist(bins=60, ax=axes[1])
axes[1].set_title('Log10 Award Amount')
axes[1].set_xlabel('log10(USD)')
plt.tight_layout()
fig.savefig(figures / '01_spend_distribution.png', dpi=120)
plt.show()

## 3. Pareto Analysis — Vendor Concentration

Interview story: *'The top 20% of vendors typically account for 80% of spend — the Pareto principle. I want to know which vendors are worth focusing the optimization effort on.'*

In [ ]:
vendor_spend = conn.execute("""
    SELECT recipient_name, SUM(action_amount) AS total_spend
    FROM federal_awards
    GROUP BY recipient_name
    ORDER BY total_spend DESC
""").df()

total = vendor_spend['total_spend'].sum()
vendor_spend['cumulative_pct'] = vendor_spend['total_spend'].cumsum() / total * 100
vendor_spend['vendor_rank'] = range(1, len(vendor_spend) + 1)
vendor_spend['vendor_pct'] = vendor_spend['vendor_rank'] / len(vendor_spend) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(vendor_spend['vendor_pct'], vendor_spend['cumulative_pct'], color='steelblue')
ax.axhline(80, color='red', linestyle='--', alpha=0.5, label='80% of spend')
pareto_20 = vendor_spend[vendor_spend['cumulative_pct'] <= 80]['vendor_pct'].max()
ax.axvline(pareto_20, color='orange', linestyle='--', alpha=0.5, label=f'Top {pareto_20:.1f}% of vendors')
ax.set_xlabel('Vendor percentile (%)')
ax.set_ylabel('Cumulative spend (%)')
ax.set_title('Pareto: Vendor Spend Concentration')
ax.legend()
fig.savefig(figures / '01_pareto_concentration.png', dpi=120)
plt.show()
print(f'Top {pareto_20:.1f}% of vendors account for 80% of spend')

## 4. Top NAICS Categories

Identify the 10–15 NAICS categories with the most spend and vendor diversity.
These are where optimization effort has the highest leverage.

In [ ]:
top_naics = get_top_naics(conn, top_n=15)
display(top_naics)
# TODO: horizontal bar chart — spend_millions by naics_description
# Save to figures/01_top_naics.png

## 5. Lead Time Distribution

Lead time = days from award action to period_of_performance_end.
Expect right-skewed; flag categories with unusually long or variable lead times.

In [ ]:
# TODO: Lead time distribution by top NAICS category (box plot)
# Flag categories with cv (std/mean) > 1 — high variability = risky procurement
# Save to figures/01_lead_time_by_naics.png

## 6. NAICS Selection for Clustering

Choose 3–5 NAICS categories to focus the analysis:
- High total spend (large opportunity)
- Multiple vendors (at least 10 — enough for clustering)
- Diverse lead times (interesting variation to explain)

Document selection and reasoning here.

In [ ]:
# TODO: Filter top_naics to categories with >= 10 vendors
# Save selection to data/processed/selected_naics.csv
SELECTED_NAICS = []  # fill in after EDA
print('Selected NAICS:', SELECTED_NAICS)

## Summary

| Finding | Value |
|---------|-------|
| Total awards | TBD |
| Unique vendors | TBD |
| Total spend | $TBD B |
| Top 20% vendor spend share | TBD% |
| Best clustering NAICS | TBD |